In [1]:
import azureml.core

from azureml.core import Workspace, Dataset, Experiment

In [2]:
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from datetime import datetime, timedelta

In [3]:
INTERVAL = 10

In [4]:
ws = Workspace.from_config()

In [5]:
df1 = pd.read_csv('data/crutcher_C_D.csv', ';')
df2 = pd.read_csv('data/tower2.csv', ';')

df1['ts'] = pd.to_datetime(df1['ts'])
df2['ts'] = pd.to_datetime(df2['ts'])

In [6]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133805 entries, 0 to 133804
Data columns (total 6 columns):
ts           133805 non-null datetime64[ns]
tag          133805 non-null object
value        133805 non-null object
machineid    133805 non-null object
siteid       133805 non-null object
lineid,      133805 non-null object
dtypes: datetime64[ns](1), object(5)
memory usage: 6.1+ MB


In [7]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203920 entries, 0 to 203919
Data columns (total 6 columns):
ts           203920 non-null datetime64[ns]
tag          203920 non-null object
value        203920 non-null object
machineid    203920 non-null object
siteid       203920 non-null object
lineid,      203920 non-null object
dtypes: datetime64[ns](1), object(5)
memory usage: 9.3+ MB


In [8]:
tags  = list(set(df1['tag']) | set(df2['tag']))

In [9]:
data  = [[0]*(len(tags)+1)] #[{tag: []} for tag in tags]

In [10]:
start = max(df1['ts'].iloc[0], df2['ts'].iloc[0]).to_pydatetime()
end   = min(df1['ts'].iloc[-1], df2['ts'].iloc[-1]).to_pydatetime()

/anaconda/envs/azureml_py36/lib/python3.6/site-packages/IPython/core/interactiveshell.py:2862: UserWarning: Discarding nonzero nanoseconds in conversion
  exec(code_obj, self.user_global_ns, self.user_ns)


In [11]:
t = start
 
new_data = []

while t < end:    
    temp1 = df1[(df1['ts'] > t) & (df1['ts'] < t + timedelta(seconds=INTERVAL))] 
    temp2 = df2[(df2['ts'] > t) & (df2['ts'] < t + timedelta(seconds=INTERVAL))]
    data  = {tag: [] for tag in tags}
    
    data['ts'] = t
    
    for tag, val in zip(temp1['tag'], temp1['value']):
        val = float(val.replace(',', '.'))
        data[tag].append(val)
    
    for tag, val in zip(temp2['tag'], temp2['value']):
        val = float(val.replace(',', '.'))
        data[tag].append(val)
    
    for tag in tags:
        if len(data[tag]):
            val = float(sum(data[tag]))/len(data[tag])
        else:
            val = 0
        data[tag] = val
    
    new_data.append(data)
    
    t = t + timedelta(seconds=INTERVAL)

In [12]:
new_df = []

for data in new_data:
    new_row = [data['ts']]
    for tag in tags:
        new_row.append(data[tag])
    new_df.append(new_row)

In [13]:
df = pd.DataFrame(new_df, columns=['ts'] + tags)

In [14]:
for column in df.columns:
    if len(set(df[column])) < 10:
        df = df.drop(columns=[column])

In [26]:
for tag in df.columns:
    df[tag] = df[tag].replace(to_replace=0, method='ffill')

In [37]:
SHIFT = 20

y = np.array(df[list(df.columns[df.columns.str.contains('Moisture')])[0]])
y = np.array([0]*SHIFT + list(y)[:-SHIFT])

df['target'] = y

In [38]:
df.head()

,ts,ControlLogix.PLC Tower 2 - Spray Dryer.PIC-14475-CV_Loop Jets Ring Pressure (CV),ControlLogix.PLC Tower 2 - Spray Dryer.TT-15031_Cyclone Temperature A2,ControlLogix.PLC Tower 2 - Spray Dryer.PT-15024-Media_Depression Average,ControlLogix.PLC Tower 2 - Burner.FIC-16602-PV_Air Flow (PV),ControlLogix.PLC Tower 2 - Spray Dryer.BC1503-Density_Density BC-1503,ControlLogix.PLC Tower 2 - Spray Dryer.TT-16612_Ring Temperature 3,ControlLogix.PLC Tower 2 - Spray Dryer.TT-15046_Cyclone Temperature B2,ControlLogix.PLC Tower 2 - Spray Dryer.TT-15054_Cyclone Temperature F2,ControlLogix.PLC Tower 2 - Spray Dryer.TT-16611_Ring Temperature 2,...,ControlLogix.PLC Tower 2 - Burner.TIC-16600-PV_Burner Temperature (PV),ControlLogix.PLC Tower 2 - Spray Dryer.TT-15038_Cyclone Temperature E1,ControlLogix.PLC Tower 2 - Spray Dryer.TT-15037_Cyclone Temperature D2,ControlLogix.PLC Tower 2 - Spray Dryer.PIC-14475-PV_Loop Jets Ring Pressure (PV),ControlLogix.PLC Tower 2 - Spray Dryer.TT-15050_Cyclone Temperature D2,ControlLogix.PLC Tower 2 - Spray Dryer.PT-15024-2_Depression Point 1,ControlLogix.PLC Slurry Making T2.WT_CrutD,ControlLogix.PLC Tower 2 - Burner.TIC-16600-SP_Burner Temperature (SP),ControlLogix.PLC Tower 2 - Spray Dryer.TT-15043_Cyclone Temperature A1,target
0,2019-08-10 09:21:55.726174,24.305920,108.249031,-14.981086,98.532188,0.0,309.046707,103.625168,105.901426,317.315173,...,325.770019,104.883271,101.235412,48.098250,107.405960,-15.103762,8.885,0.0,100.453964,0.0
1,2019-08-10 09:22:05.726174,24.291950,108.249031,-15.098358,98.731605,529.0,310.009715,103.690018,105.998700,317.801575,...,326.393204,105.758754,101.322960,48.080413,107.470825,-15.233463,8.915,0.0,100.389106,0.0
2,2019-08-10 09:22:15.726174,24.455382,108.346306,-15.370190,97.774540,529.0,310.972770,103.722443,105.998700,318.093399,...,327.049712,106.134887,101.322960,47.905317,107.503250,-15.520428,8.915,0.0,100.518814,0.0
3,2019-08-10 09:22:25.726174,24.694814,108.638130,-15.364786,98.392055,529.0,311.838523,103.787293,106.063553,318.463026,...,327.936233,106.517509,101.614788,47.714010,107.503250,-15.392347,8.955,0.0,100.551239,0.0
4,2019-08-10 09:22:35.726174,25.024079,108.929962,-15.103760,98.348380,529.0,312.509723,103.904022,106.063553,318.754851,...,328.600388,106.712059,101.712067,47.457848,107.665375,-15.201036,8.945,0.0,100.583664,0.0


In [39]:
os.makedirs('to_upload', exist_ok=True)
df.to_csv('to_upload/dataset.csv', index=False)

In [40]:
dstore = ws.get_default_datastore()

In [41]:
dstore.upload('to_upload', 'unilever/datasets', overwrite=True)

Uploading an estimated of 2 files
Uploading to_upload/.ipynb_checkpoints/dataset-checkpoint.csv
Uploading to_upload/dataset.csv
Uploaded to_upload/.ipynb_checkpoints/dataset-checkpoint.csv, 1 files out of an estimated total of 2
Uploaded to_upload/dataset.csv, 2 files out of an estimated total of 2
Uploaded 2 files


$AZUREML_DATAREFERENCE_de7c9f6a80f84f45aeb3f0c865d5cc16

In [42]:
dset = Dataset.Tabular.from_delimited_files(dstore.path('unilever/datasets/dataset.csv'))

In [44]:
dset.register(ws, 'indaiatuba2', create_new_version=True)

TabularDataset
{
  "source": {
    "datastores": [
      {
        "subscription": "60582a10-b9fd-49f1-a546-c4194134bba8",
        "resourceGroup": "copetersrg",
        "workspaceName": "azure-ml",
        "datastoreName": "workspaceblobstore",
        "path": "unilever/datasets/dataset.csv"
      }
    ]
  },
  "definition": [
    "GetDatastoreFiles",
    "ParseDelimited",
    "DropColumns",
    "SetColumnTypes"
  ],
  "registration": {
    "name": "indaiatuba2",
    "version": 1,
    "description": null,
    "tags": {},
    "id": null,
    "workspaceName": "azure-ml",
    "resourceGroup": "copetersrg",
    "subscription": "60582a10-b9fd-49f1-a546-c4194134bba8"
  }
}

In [22]:
dset

TabularDataset
{
  "source": {
    "datastores": [
      {
        "subscription": "60582a10-b9fd-49f1-a546-c4194134bba8",
        "resourceGroup": "copetersrg",
        "workspaceName": "azure-ml",
        "datastoreName": "workspaceblobstore",
        "path": "unilever/datasets/dataset.csv"
      }
    ]
  },
  "definition": [
    "GetDatastoreFiles",
    "ParseDelimited",
    "DropColumns",
    "SetColumnTypes"
  ],
  "registration": null
}